# Beyond Serialization: Evidence-Certified Graph Reasoning

**Author: Keedar Kale** · Google DeepMind Gemma 4 Developer Agent Paper Track

This notebook is a **self-contained, internet-free research demonstration**. It reconstructs the open-source experimental Python implementation inside a temporary working directory, generates 16 miniature buggy Python projects, and reproduces the paper's diagnostic metrics. It needs **no GPU, no installed Gemma 4 weights, and no external APIs**.

**Scope:** This studies static code localization and inspectable graph paths. It does **not** run Gemma 4, train a GNN, automatically repair bugs, or evaluate against hidden Kaggle tasks.

In [ ]:
from pathlib import Path
import json, sys

ROOT = Path.cwd() / 'ecwg_research_notebook'
ROOT.mkdir(exist_ok=True)
print('Research workspace:', ROOT)

## 1. Reconstruct the reproducible experiment

The following cell contains the same stdlib-only Python source used by the companion GitHub repository. Including it here makes the notebook independently runnable even if the network is disabled. The code creates all toy cases; it does not access private competition labels or patches.

In [ ]:
SOURCE_FILES = {'src/ecwg/__init__.py': '"""Evidence-Certified Witness Graphs: offline, test-conditioned code localization."""\nfrom .core import analyze, rank, tokens, get_reachable\nfrom .certificates import make_certificate, verify_certificate, shortest_path\n\n__version__ = "0.1.0"\n__all__ = ["analyze", "rank", "tokens", "get_reachable", "make_certificate", "verify_certificate", "shortest_path"]\n', 'src/ecwg/core.py': '#!/usr/bin/env python3\n"""WitnessGraph pilot: stdlib-only, offline test-conditioned static code localization.\n\nResearch demonstration, NOT a Gemma 4 agent or competition-scored implementation.\nGraph uses AST-resolved imports and function calls. It is an imperfect static proxy,\nnot a causal execution trace. All scoring rules are fixed before benchmarking.\n"""\nfrom __future__ import annotations\nimport argparse, ast, json, math, re\nfrom collections import defaultdict, deque\nfrom pathlib import Path\n\n\ndef tokens(s: str) -> set[str]:\n    s = re.sub(r\'([a-z0-9])([A-Z])\', r\'\\1 \\2\', s)\n    return set(re.findall(r\'[a-z0-9]+\', s.lower().replace(\'_\', \' \'))) - {\n        \'the\', \'a\', \'an\', \'on\', \'for\', \'at\', \'and\', \'to\', \'of\', \'in\', \'with\', \'when\',\n        \'should\', \'test\', \'regression\', \'value\', \'return\', \'def\', \'int\', \'from\',\n        \'import\', \'is\', \'not\', \'be\', \'correct\', \'wrong\', \'must\', \'its\', \'by\',\n    }\n\n\ndef analyze(root: Path) -> tuple[dict, dict]:\n    root = Path(root)\n    nodes, tree_cache, imports_cache = {}, {}, {}\n    for path in root.rglob(\'*.py\'):\n        if any(x in path.parts for x in (\'__pycache__\', \'.venv\', \'venv\')):\n            continue\n        rel = str(path.relative_to(root)).replace(\'\\\\\', \'/\')\n        mod = rel[:-3].replace(\'/\', \'.\')\n        if mod.endswith(\'.__init__\'):\n            mod = mod[:-9]\n        try:\n            source = path.read_text(encoding=\'utf-8\')\n            tree = ast.parse(source)\n        except (UnicodeError, SyntaxError):\n            continue\n        tree_cache[rel] = (mod, tree, source)\n        import_map = {}\n        for statement in tree.body:\n            if isinstance(statement, ast.ImportFrom):\n                base = (statement.module or \'\')\n                if statement.level:\n                    prefix = mod.split(\'.\')[:-statement.level]\n                    base = \'.\'.join(prefix + ([base] if base else []))\n                for item in statement.names:\n                    if item.name != \'*\':\n                        import_map[item.asname or item.name] = base + \'.\' + item.name\n            elif isinstance(statement, ast.Import):\n                for item in statement.names:\n                    import_map[item.asname or item.name.split(\'.\')[0]] = item.name\n        imports_cache[rel] = import_map\n        for statement in tree.body:\n            if isinstance(statement, (ast.FunctionDef, ast.AsyncFunctionDef)):\n                fqn = mod + \'.\' + statement.name\n                snippet = ast.get_source_segment(source, statement) or statement.name\n                nodes[fqn] = {\'file\': rel, \'name\': statement.name,\n                              \'text\': snippet[:3000], \'is_test\': rel.startswith(\'tests/\')}\n    edges = defaultdict(set)\n    for rel, (mod, tree, _) in tree_cache.items():\n        import_map = imports_cache[rel]\n        for fn in tree.body:\n            if not isinstance(fn, (ast.FunctionDef, ast.AsyncFunctionDef)):\n                continue\n            src = mod + \'.\' + fn.name\n            if src not in nodes:\n                continue\n            for child in ast.walk(fn):\n                if not isinstance(child, ast.Call):\n                    continue\n                name = None\n                if isinstance(child.func, ast.Name):\n                    name = import_map.get(child.func.id, mod + \'.\' + child.func.id)\n                elif isinstance(child.func, ast.Attribute) and isinstance(child.func.value, ast.Name):\n                    base = child.func.value.id\n                    name = import_map.get(base, base) + \'.\' + child.func.attr\n                if name in nodes and name != src:\n                    edges[src].add(name)\n    return nodes, edges\n\n\ndef get_reachable(edges: dict, test_symbol: str, max_depth: int = 4) -> dict:\n    distances = {test_symbol: 0}\n    queue = deque([test_symbol])\n    while queue:\n        node = queue.popleft()\n        if distances[node] == max_depth:\n            continue\n        for child in edges.get(node, ()):\n            if child not in distances:\n                distances[child] = distances[node] + 1\n                queue.append(child)\n    return distances\n\n\ndef rank(root: Path, issue: str, failing_test_symbol: str, method: str = \'witness\') -> list[dict]:\n    """Rank production functions without using any ground-truth label or patch."""\n    nodes, edges = analyze(root)\n    q = tokens(issue + \' \' + failing_test_symbol)\n    products = {key: node for key, node in nodes.items() if not node[\'is_test\']}\n    raw = {}\n    for key, node in products.items():\n        # Deliberately elementary lexical baseline, not a fitted BM25 implementation.\n        names = tokens(key.replace(\'.\', \' \') + \' \' + (ast.get_docstring(\n            ast.parse(node[\'text\']).body[0]) or \'\' if node[\'text\'].startswith((\'def \', \'async def \')) else \'\'))\n        raw[key] = len(q & names) / math.sqrt(max(1, len(names)))\n    max_raw = max(raw.values(), default=1.0) or 1.0\n    reachable = get_reachable(edges, failing_test_symbol)\n    rows = []\n    for key, node in products.items():\n        lex = raw[key] / max_raw\n        on_path = key in reachable and reachable[key] > 0\n        # A structural terminal is a node with no observed local outgoing function calls.\n        terminal = on_path and len(edges.get(key, ())) == 0\n        score = lex if method == \'lexical\' else 0.25 * lex + 0.20 * on_path + 0.55 * terminal\n        rows.append({\'symbol\': key, \'file\': node[\'file\'], \'score\': round(float(score), 6),\n                     \'lexical\': round(float(lex), 6), \'on_test_path\': bool(on_path),\n                     \'terminal\': bool(terminal), \'distance\': reachable.get(key)})\n    rows.sort(key=lambda x: (-x[\'score\'], x[\'symbol\']))\n    return rows\n\n\n', 'src/ecwg/certificates.py': '"""Static graph evidence certificates.\n\nA certificate proves only that a claimed path exists in the AST-derived graph\nand that the referenced source files have not changed since it was created.\nIt cannot establish that a patch is correct or that an edge executed at runtime.\n"""\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom hashlib import sha256\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .core import analyze\n\nSCHEMA = \'ecwg.static-call-path.v1\'\n\n\ndef shortest_path(edges: dict[str, set[str]], start: str, end: str) -> list[str] | None:\n    if start == end:\n        return [start]\n    seen = {start}\n    todo = deque([(start, [start])])\n    while todo:\n        node, path = todo.popleft()\n        for nxt in sorted(edges.get(node, ())):\n            if nxt in seen:\n                continue\n            if nxt == end:\n                return path + [nxt]\n            seen.add(nxt)\n            todo.append((nxt, path + [nxt]))\n    return None\n\n\ndef _digest(root: Path, rel_path: str) -> str:\n    # Each relative path is derived from our Python AST scanner, never user input.\n    absolute = (root / rel_path).resolve()\n    absolute.relative_to(root.resolve())\n    return sha256(absolute.read_bytes()).hexdigest()\n\n\ndef make_certificate(root: Path, failing_test: str, target: str) -> dict[str, Any]:\n    """Record an auditable directed test-to-target path and file hashes."""\n    root = Path(root)\n    nodes, edges = analyze(root)\n    if failing_test not in nodes or not nodes[failing_test][\'is_test\']:\n        raise ValueError(f\'Not a discovered test function: {failing_test}\')\n    if target not in nodes or nodes[target][\'is_test\']:\n        raise ValueError(f\'Not a discovered production function: {target}\')\n    path = shortest_path(edges, failing_test, target)\n    if not path or len(path) < 2:\n        raise ValueError(\'No directed call path from failing test to target\')\n    file_list = sorted({nodes[symbol][\'file\'] for symbol in path})\n    return {\n        \'schema\': SCHEMA,\n        \'evidence\': \'static_ast_call_graph_not_runtime_or_patch_proof\',\n        \'failing_test\': failing_test,\n        \'target\': target,\n        \'path\': path,\n        \'files_sha256\': {file: _digest(root, file) for file in file_list},\n    }\n\n\ndef verify_certificate(root: Path, certificate: dict[str, Any]) -> tuple[bool, list[str]]:\n    """Verify path edges, endpoint roles, and all source hashes.\n\n    Never treat certificates as trusted signatures: an attacker may rewrite both\n    source and hashes. This detects stale/tampered files against a given record.\n    """\n    root = Path(root)\n    errors: list[str] = []\n    nodes, edges = analyze(root)\n    if certificate.get(\'schema\') != SCHEMA:\n        errors.append(\'unsupported_schema\')\n    path = certificate.get(\'path\')\n    if not isinstance(path, list) or len(path) < 2 or any(not isinstance(x, str) for x in path):\n        return False, errors + [\'invalid_path\']\n    start = certificate.get(\'failing_test\')\n    target = certificate.get(\'target\')\n    if path[0] != start or path[-1] != target:\n        errors.append(\'endpoint_mismatch\')\n    if start not in nodes or not nodes[start][\'is_test\']:\n        errors.append(\'test_not_found\')\n    if target not in nodes or nodes[target][\'is_test\']:\n        errors.append(\'production_target_not_found\')\n    for source, destination in zip(path, path[1:]):\n        if source not in nodes or destination not in edges.get(source, ()):\n            errors.append(f\'missing_directed_edge:{source}->{destination}\')\n    hashes = certificate.get(\'files_sha256\')\n    if not isinstance(hashes, dict):\n        return False, errors + [\'hashes_missing\']\n    expected_files = {nodes[s][\'file\'] for s in path if s in nodes}\n    if set(hashes) != expected_files:\n        errors.append(\'file_set_mismatch\')\n    for file in sorted(expected_files):\n        try:\n            if hashes.get(file) != _digest(root, file):\n                errors.append(f\'changed_file:{file}\')\n        except (OSError, ValueError):\n            errors.append(f\'unreadable_file:{file}\')\n    return (not errors), errors\n', 'scripts/run_pilot.py': '#!/usr/bin/env python3\n"""Create/re-run a 16-case, fully reproducible synthetic cross-file stress test.\n\nTwelve deep-implementation faults and four entrypoint faults are intentionally\nincluded. The mixture is NOT a representative prevalence estimate.\n"""\nfrom __future__ import annotations\nimport json, shutil, subprocess, sys\nfrom pathlib import Path\nfrom ecwg import rank\n\nROOT = Path(__file__).resolve().parents[1]\nFIX = ROOT / \'data\' / \'synthetic_cases\'\nDOMAINS = [\n    \'invoice_rounding\', \'session_expiry\', \'retry_backoff\', \'cache_invalidation\',\n    \'cursor_pagination\', \'unicode_escaping\', \'request_throttling\', \'image_scaling\',\n    \'search_deduplication\', \'config_precedence\', \'timezone_scheduling\',\n    \'percentile_metrics\', \'shipment_costs\', \'stock_reservation\',\n    \'upload_checksum\', \'notification_batching\',\n]\n\n\ndef create_case(i: int, domain: str):\n    case = FIX / f\'case_{i+1:02d}\'\n    if case.exists():\n        shutil.rmtree(case)\n    for folder in (\'lib\', \'tests\'):\n        (case / folder).mkdir(parents=True)\n        (case / folder / \'__init__.py\').write_text(\'\', encoding=\'utf-8\')\n    deep = i < 12\n    fname = f\'apply_{domain}\'\n    issue = f\'The {domain.replace("_", " ")} path returns the wrong numerical result under a regression scenario.\'\n    api = (f\'from lib.engine_{i+1:02d} import calculate\\n\\n\'\n           f\'def {fname}(value: int) -> int:\\n\'\n           f\'    """Apply {domain.replace("_", " ")} for this request."""\\n\'\n           f\'    return calculate(value){"" if deep else " - 1"}\\n\')\n    engine = (f\'def calculate(value: int) -> int:\\n\'\n              f\'    """Perform a low-level numerical operation."""\\n\'\n              f\'    return value + {1 if deep else 2}\\n\')\n    legacy = (f\'def {fname}_legacy(value: int) -> int:\\n\'\n              f\'    """Obsolete {domain.replace("_", " ")} utility. Do not call."""\\n\'\n              f\'    return value + 3\\n\')\n    test_name = f\'test_{domain}_regression\'\n    test = (f\'from lib.{domain}_api import {fname}\\n\\n\'\n            f\'def {test_name}():\\n\'\n            f\'    assert {fname}(3) == 5\\n\')\n    (case / \'lib\' / f\'{domain}_api.py\').write_text(api, encoding=\'utf-8\')\n    (case / \'lib\' / f\'engine_{i+1:02d}.py\').write_text(engine, encoding=\'utf-8\')\n    (case / \'lib\' / f\'{domain}_legacy.py\').write_text(legacy, encoding=\'utf-8\')\n    (case / \'tests\' / f\'test_{domain}.py\').write_text(test, encoding=\'utf-8\')\n    target = f\'lib.engine_{i+1:02d}.calculate\' if deep else f\'lib.{domain}_api.{fname}\'\n    bad_path = case / \'lib\' / (f\'engine_{i+1:02d}.py\' if deep else f\'{domain}_api.py\')\n    original = bad_path.read_text(encoding=\'utf-8\')\n    replacement = original.replace(\'return value + 1\', \'return value + 2\') if deep else original.replace(\'return calculate(value) - 1\', \'return calculate(value)\')\n    def check():\n        for cache in case.rglob("__pycache__"):\n            shutil.rmtree(cache)\n        return subprocess.run([sys.executable, \'-c\', f\'from tests.test_{domain} import {test_name}; {test_name}()\'],\n                              cwd=case, capture_output=True, text=True, timeout=20).returncode\n    # Confirm failing test and ground-truth correction; restore original afterwards.\n    before = check()\n    bad_path.write_text(replacement, encoding=\'utf-8\')\n    after = check()\n    bad_path.write_text(original, encoding=\'utf-8\')\n    if before == 0 or after != 0:\n        raise RuntimeError(f\'Invalid fixture {domain}: failing={before}, fixed={after}\')\n    failing = f\'tests.test_{domain}.{test_name}\'\n    bas = rank(case, issue, failing, \'lexical\')\n    wit = rank(case, issue, failing, \'witness\')\n    return {\'id\': f\'case_{i+1:02d}\', \'issue\': issue, \'failing_test\': failing,\n            \'fault_depth\': \'implementation\' if deep else \'entrypoint\',\n            \'ground_truth_symbol\': target, \'test_fails_before_fix\': True,\n            \'test_passes_after_fix\': True,\n            \'lexical_rank\': next((j+1 for j, x in enumerate(bas) if x[\'symbol\']==target), None),\n            \'witness_rank\': next((j+1 for j, x in enumerate(wit) if x[\'symbol\']==target), None),\n            \'lexical_top1\': bas[0][\'symbol\'], \'witness_top1\': wit[0][\'symbol\']}\n\n\ndef main():\n    FIX.mkdir(exist_ok=True, parents=True)\n    (ROOT / \'results\').mkdir(exist_ok=True)\n    rows = [create_case(i, domain) for i, domain in enumerate(DOMAINS)]\n    def metrics(key):\n        return {\'top1_correct\': sum(x[key] == 1 for x in rows),\n                \'top1_total\': len(rows),\n                \'recall_at_2\': sum(x[key] is not None and x[key] <= 2 for x in rows)}\n    summary = {\'dataset\': \'16 deliberately designed synthetic multi-file Python bugs\',\n               \'composition\': {\'implementation_level\': 12, \'entrypoint_level\': 4},\n               \'baseline\': \'Lexical symbol/docstring overlap + failing-test name\',\n               \'method\': \'Fixed-score test-conditioned AST call-graph routing\',\n               \'lexical\': metrics(\'lexical_rank\'),\n               \'witnessgraph\': metrics(\'witness_rank\'),\n               \'notes\': \'This is a biased synthetic stress test, not competition or real-world issue resolution performance.\',\n               \'cases\': rows}\n    (ROOT / \'results\' / \'pilot_results.json\').write_text(json.dumps(summary, indent=2), encoding=\'utf-8\')\n    print(json.dumps({k:v for k,v in summary.items() if k != \'cases\'}, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/representation_robustness.py': '#!/usr/bin/env python3\n"""Small, reproducible robustness and certificate sanity checks.\n\nImportant: this is a deterministic diagnostic over synthetic repos, not an LLM,\nGNN, Kaggle held-out benchmark, or real bug fix evaluation.\n"""\nfrom __future__ import annotations\nimport json, shutil, tempfile\nfrom collections import deque\nfrom pathlib import Path\nfrom run_pilot import main as generate_pilot\nfrom ecwg import analyze, rank\n\nHERE = Path(__file__).resolve().parents[1]\n\n\ndef find_path(edges, start, goal):\n    if start == goal: return [start]\n    q = deque([(start, [start])]); seen = {start}\n    while q:\n        at, p = q.popleft()\n        for nxt in sorted(edges.get(at, ())):\n            if nxt in seen: continue\n            if nxt == goal: return p + [nxt]\n            seen.add(nxt); q.append((nxt, p + [nxt]))\n    return None\n\n\ndef certificate_valid(nodes, edges, path):\n    return bool(path and len(path) >= 2 and all(n in nodes for n in path)\n                and all(v in edges.get(u, ()) for u, v in zip(path, path[1:])))\n\n\ndef rename_low_level_calculate(repo):\n    # Semantics-preserving source rewrite: update one imported function consistently.\n    for path in repo.rglob(\'*.py\'):\n        text = path.read_text(encoding=\'utf-8\')\n        text = text.replace(\'calculate\', \'compute_value\')\n        path.write_text(text, encoding=\'utf-8\')\n\n\ndef evaluate():\n    generate_pilot()\n    base = json.loads((HERE / \'results\' / \'pilot_results.json\').read_text(encoding=\'utf-8\'))\n    outcomes = []\n    with tempfile.TemporaryDirectory(prefix=\'witness_graph_\') as temporary:\n        temp = Path(temporary)\n        for rec in base[\'cases\']:\n            original = HERE / \'data\' / \'synthetic_cases\' / rec[\'id\']\n            case = temp / rec[\'id\']; shutil.copytree(original, case)\n            issue, failing, target = rec[\'issue\'], rec[\'failing_test\'], rec[\'ground_truth_symbol\']\n            nodes, edges = analyze(case)\n            valid = find_path(edges, failing, target)\n            # Tamper with the last step: a real node is substituted, but the call edge is absent.\n            legacy = next(name for name in nodes if name.endswith(\'_legacy\'))\n            fake = valid[:-1] + [legacy] if valid else [failing, legacy]\n            baseline_rank = rec[\'witness_rank\']\n            # Ablation removes the failing-test symbol; this disables witness reachability.\n            witness_blind = rank(case, issue, \'\', method=\'witness\')\n            blind_position = next((j+1 for j, r in enumerate(witness_blind)\n                                   if r[\'symbol\'] == target), None)\n            # Identifier perturbation - target and call sites renamed consistently.\n            rename_low_level_calculate(case)\n            renamed_target = target.replace(\'calculate\', \'compute_value\')\n            renamed = rank(case, issue, failing, method=\'witness\')\n            renamed_position = next((j+1 for j, r in enumerate(renamed)\n                                     if r[\'symbol\'] == renamed_target), None)\n            outcomes.append({\n                \'id\': rec[\'id\'], \'depth\': rec[\'fault_depth\'],\n                \'original_witness_rank\': baseline_rank,\n                \'no_test_witness_rank\': blind_position,\n                \'renamed_witness_rank\': renamed_position,\n                \'genuine_certificate_valid\': certificate_valid(nodes, edges, valid),\n                \'tampered_certificate_rejected\': not certificate_valid(nodes, edges, fake),\n                \'certificate_length_edges\': len(valid)-1 if valid else None,\n            })\n    top1 = lambda key: sum(r[key] == 1 for r in outcomes)\n    summary = {\n        \'experimental_scope\': \'16 constructed Python fixtures; no LLM, GNN, or Kaggle scoring\',\n        \'synthetic_case_count\': len(outcomes),\n        \'baseline_lexical_top1\': base[\'lexical\'][\'top1_correct\'],\n        \'baseline_witness_top1\': top1(\'original_witness_rank\'),\n        \'witness_without_test_id_top1\': top1(\'no_test_witness_rank\'),\n        \'witness_after_consistent_identifier_rename_top1\': top1(\'renamed_witness_rank\'),\n        \'genuine_paths_validated\': sum(r[\'genuine_certificate_valid\'] for r in outcomes),\n        \'tampered_paths_rejected\': sum(r[\'tampered_certificate_rejected\'] for r in outcomes),\n        \'cautions\': [\'Synthetic fixture distribution selected deliberately\',\n                     \'Certificate check verifies graph edges, not causal influence\',\n                     \'Renaming test rewrites exactly the low-level calculate symbol\',\n                     \'No learned GNN or Gemma model was tested\'],\n        \'cases\': outcomes,\n    }\n    (HERE / \'results\' / \'proposal_diagnostics.json\').write_text(json.dumps(summary, indent=2), encoding=\'utf-8\')\n    print(json.dumps({k: v for k, v in summary.items() if k != \'cases\'}, indent=2))\n\nif __name__ == \'__main__\':\n    evaluate()\n'}
for relative, source in SOURCE_FILES.items():
    filename = ROOT / relative
    filename.parent.mkdir(parents=True, exist_ok=True)
    filename.write_text(source, encoding='utf-8')
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
print('Written', len(SOURCE_FILES), 'source modules/scripts')

## 2. Run the full synthetic evaluation

We generate 16 failing cases (12 faults in a low-level function, four in a wrapper). Each case's regression assertion is executed before and after a known repair, and the faulty source is restored. The baseline and graph ranker then receive the same issue and failing-test identity, **without the ground-truth target**.

The experiment also checks identifier-renaming robustness, the missing-test-ID ablation, and directed-path validity.

In [ ]:
from representation_robustness import evaluate

evaluate()
result = json.loads((ROOT/'results/pilot_results.json').read_text(encoding='utf-8'))
extra = json.loads((ROOT/'results/proposal_diagnostics.json').read_text(encoding='utf-8'))
print('Lexical top-1:', result['lexical']['top1_correct'], '/', result['lexical']['top1_total'])
print('Graph top-1:  ', result['witnessgraph']['top1_correct'], '/', result['witnessgraph']['top1_total'])
print('Graph recall@2:', result['witnessgraph']['recall_at_2'], '/', result['witnessgraph']['top1_total'])
print('Extra diagnostic metrics:')
print(json.dumps({k:v for k,v in extra.items() if k!='cautions'}, indent=2))

## 3. Inspect individual cases and verify a graph witness

A witness certificate records the sequence of statically discovered function calls and SHA-256 hashes for the source files it references. This is checkable evidence of a **static source path**, not proof of runtime causality or a successful patch.

In [ ]:
from ecwg import rank, make_certificate, verify_certificate
case = result['cases'][0]
repo = ROOT/'data/synthetic_cases'/case['id']
rows = rank(repo, case['issue'], case['failing_test'])
print('Ranked candidates:')
for r in rows[:4]:
    print(r['symbol'], 'score=', r['score'], 'on_path=', r['on_test_path'])

certificate = make_certificate(repo, case['failing_test'], case['ground_truth_symbol'])
print('\nCertificate:')
print(json.dumps(certificate, indent=2))
print('Verified:', verify_certificate(repo, certificate))

## 4. Per-case results and limitations

The observed 75% top-1 is **not** a generalization claim: the 16-case sample was deliberately constructed, and the graph method favors terminal nodes. All four wrapper-level faults fail at top-1. The following table makes that behavior visible.

In [ ]:
print(f"{'Case':8} {'Fault type':17} {'Lexical rank':13} {'Graph rank':10}")
print('-'*54)
for r in result['cases']:
    print(f"{r['id']:8} {r['fault_depth']:17} {r['lexical_rank']:13} {r['witness_rank']:10}")
print('\nSynthetic pilot only. No Gemma 4 patch-generation measurements.')

## 5. Research next steps (proposed, not executed)

- Hold the model, hardware, memory, and context budgets fixed while comparing serialized graph text, relevant retrieved graph paths, GNN-encoded structure, and hybrids.
- Test on **real held-out repositories** with independently scored fix outcomes.
- Separate retrieval/path correctness from patch quality and test pass rates.
- Report results across hop counts, graph size, edge ordering, name perturbations, and multiple seeds.

**Companion paper:** *Beyond Serialization: Evidence-Certified Graph Reasoning for Offline Gemma 4 Software Agents* — Keedar Kale. Publish this notebook publicly on Kaggle and add its actual URL to your Kaggle Paper Track Writeup.